In [22]:
from datasets import load_dataset
import torch
def LoadData(DatasetID) :
    TrainData=load_dataset(
        DatasetID,
        split="train",
        streaming=True
    )
    ValData=load_dataset(
         DatasetID,
         split="train",
         streaming=True
    )
    TrainStories=[story["text"] for story in TrainData.take(2)]
    ValStories=[story["text"] for story in ValData.take(9000)]
    return TrainStories ,ValStories


DATASET_ID = "skeskinen/TinyStories-hf"
T,V=LoadData(DATASET_ID)

In [ ]:
import tiktoken 
tokenizer = tiktoken.get_encoding("gpt2")
text="go fuck your self"

token_ids=tokenizer.encode(V[0])
decoded=tokenizer.decode(token_ids)



[3198, 1110, 11, 257, 1310, 2576, 3706, 20037, 1043, 257, 17598, 287, 607, 2119, 13, 1375, 2993, 340, 373, 2408, 284, 711, 351, 340, 780, 340, 373, 7786, 13, 20037, 2227, 284, 2648, 262, 17598, 351, 607, 1995, 11, 523, 673, 714, 34249, 257, 4936, 319, 607, 10147, 13, 198, 198, 43, 813, 1816, 284, 607, 1995, 290, 531, 11, 366, 29252, 11, 314, 1043, 428, 17598, 13, 1680, 345, 2648, 340, 351, 502, 290, 34249, 616, 10147, 1701, 2332, 1995, 13541, 290, 531, 11, 366, 5297, 11, 20037, 11, 356, 460, 2648, 262, 17598, 290, 4259, 534, 10147, 526, 198, 198, 41631, 11, 484, 4888, 262, 17598, 290, 384, 19103, 262, 4936, 319, 20037, 338, 10147, 13, 632, 373, 407, 2408, 329, 606, 780, 484, 547, 7373, 290, 5742, 1123, 584, 13, 2293, 484, 5201, 11, 20037, 26280, 607, 1995, 329, 7373, 262, 17598, 290, 18682, 607, 10147, 13, 1119, 1111, 2936, 3772, 780, 484, 550, 4888, 290, 3111, 1978, 13]
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it w

In [ ]:
import torch 
from torch import nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self , d_model=128 , num_heads=4 ):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model=d_model
        self.num_heads=num_heads
        self.projections=nn.Linear(d_model,3*d_model)
        self.OutputProj=nn.LazyLinear(d_model)


    def forward(self,x) :
        B ,SL ,C=x.size()
        Projections=self.projections(x)
        Q,K,V=Projections.split(self.d_model , 2)

        Q=Q.reshape(B,SL,self.num_heads , -1).transpose(1,2)
        K=K.reshape(B,SL,self.num_heads , -1).transpose(1,2)
        V=V.reshape(B,SL,self.num_heads , -1).transpose(1,2)

        AttentionOutput=F.scaled_dot_product_attention(Q,K,V , is_causal=True,dropout_p=0.0)

        AttentionOutput=AttentionOutput.transpose(1,2)
        AttentionOutput=AttentionOutput.reshape(B , SL , -1)

        Attention=self.OutputProj(AttentionOutput)
        return Attention



torch.manual_seed(42)

model = CausalSelfAttention(
    d_model=128,
    num_heads=4
)

x = torch.randn(2, 6, 128)

# Test 1: Output shape
output = model(x)
assert output.shape == (2, 6, 128)

# Test 2: Finite values
assert torch.isfinite(output).all()

# Test 3: Causal independence
model.eval()

with torch.no_grad():
    output1 = model(x)

    modified = x.clone()
    modified[:, 5, :] += 100

    output2 = model(modified)

assert torch.allclose(
    output1[:, :5, :],
    output2[:, :5, :],
    atol=1e-5
)

# Test 4: Gradient propagation
model.train()
output = model(x)
output.square().mean().backward()

assert all(
    p.grad is not None
    for p in model.parameters()
)

print("CausalSelfAttention passed!")


CausalSelfAttention passed!


In [ ]:

import torch 
from torch import nn 
from src.transformer import GptBlock


class MiniGpt (nn.Module) :
    def __init__(self,vocab_size ,d_model,context_size ,num_heads,d_ff,num_layers) :
        super().__init__()
        self.transformer=nn.ModuleDict(dict(
            Embeddings=nn.Embedding(vocab_size , d_model),
            PositionalEmbeddings=nn.Embedding(context_size,d_model),
            Blocks=nn.ModuleList([GptBlock(d_model ,num_heads ,d_ff) for _ in range(num_layers)]),
            LayerNorm=nn.LayerNorm(d_model),
        ))
        self.modelhead=nn.Linear(d_model,vocab_size,bias=False)
        self.modelhead.weight = self.transformer.Embeddings.weight
        self.vocab_size=vocab_size
        self.d_model=d_model
        self.context_size=context_size
        self.num_heads=num_heads
        self.d_ff=d_ff
        self.num_layers=num_layers

    def forward(self ,tokens) :
        Batch,Context=tokens.size()
        assert self.context_size >= Context ,f"max context size={self.context_size} ,cuurent sequence lenght is {Context}"
        Positions=torch.arange(0,Context,dtype=torch.long,device=tokens.device)
        PositionsEmbedding=self.transformer.PositionalEmbeddings(Positions)
        Embeddings=self.transformer.Embeddings(tokens)
        x=Embeddings+PositionsEmbedding
        for block in self.transformer.Blocks :
            x=block(x)
        x=self.transformer.LayerNorm(x)
        logits=self.modelhead(x)

        return logits   

In [15]:
import src
from src.model import Scheduler ,MiniGpt
import scripts
import torch
import os

torch.manual_seed(42)

# 1. Initialize a small model
model = MiniGpt(
    vocab_size=100,
    d_model=64,
    context_size=32,
    num_heads=4,
    d_ff=256,
    num_layers=2
)

# 2. Initialize optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

# 3. Initialize scheduler
scheduler = Scheduler(
    optimizer,
    warmupiter=2,
    maxiter=10
)

# 4. Generate synthetic token data
train_data = torch.randint(0, 100, (5000,))
val_data = torch.randint(0, 100, (1000,))

# 5. Run 10 training steps
train_losses, val_losses = model.train_model(
    optimizer,
    train_data,
    val_data,
    eval_interval=5,
    eval_steps=2,
    schedular=scheduler,
    batch_size=4,
    block_size=16,
    max_steps=10
)

# 6. Verify results
assert len(train_losses) == 10
assert len(val_losses) == 2

assert all(
    torch.isfinite(torch.tensor(train_losses))
)

assert os.path.isfile("./checkpoints/best_model.pt")

print("Training pipeline passed!")
print("Final training loss:", train_losses[-1])
print("Validation losses:", val_losses)


validation loss is 42.80393600463867
train Loss is 42.68708801269531
validation loss is 39.854597091674805
Training pipeline passed!
Final training loss: 42.68708801269531
Validation losses: [42.80393600463867, 39.854597091674805]
